In [1]:
!pip install torch transformers tree_sitter==0.21.3 scikit-learn tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 502.2/502.2 kB 15.4 MB/s eta 0:00:00


In [2]:
import os
import json
import math
import torch
import logging
import random
import numpy as np
from tqdm import tqdm
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import CrossEntropyLoss
from torch.amp import autocast, GradScaler
from torch.utils.data import DataLoader, Dataset, RandomSampler, SequentialSampler, Subset
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup, RobertaConfig, RobertaModel, AutoTokenizer
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, roc_auc_score, average_precision_score, classification_report, confusion_matrix
from collections import defaultdict, Counter

class Args:
    output_dir = "saved_models"
    model_name_or_path = "microsoft/graphcodebert-base"
    train_file = "/kaggle/input/datasets/hasanmahmudabdullah/dfgdataset2/dataset_graphcodebert.jsonl"

    code_length = 384
    data_flow_length = 128

    # backward compatibility with older cells
    codelength = code_length
    dataflow_length = data_flow_length

    train_batch_size = 16
    eval_batch_size = 32
    learning_rate = 2e-5
    max_grad_norm = 1.0
    num_train_epochs = 5
    patience = 2
    metric_for_best_model = "accuracy"
    best_model_path = os.path.join(output_dir, "best_model.bin")
    split_indices_path = os.path.join(output_dir, "graphcodebert_split_indices.json")
    split_summary_path = os.path.join(output_dir, "graphcodebert_split_summary.json")
    seed = 42
    test_ratio = 0.10
    val_ratio = 0.08
    num_workers = 2
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
args = Args()
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(args.seed)


In [3]:
class Model(nn.Module):   
    def __init__(self, encoder, config):
        super(Model, self).__init__()
        self.encoder = encoder
        self.config = config
        self.dropout = nn.Dropout(config.hidden_dropout_prob)
        self.classifier = nn.Linear(config.hidden_size, 2)

    def forward(self, input_ids=None, p_ids=None, attn_mask=None, labels=None): 
        # Convert mask: 1 -> 0 (attend), 0 -> -10000 (mask)
        extended_attention_mask = (1.0 - attn_mask) * -10000.0
        extended_attention_mask = extended_attention_mask.unsqueeze(1)

        # Get Embeddings using p_ids and input_ids directly
        embedding_output = self.encoder.embeddings(
            input_ids=input_ids, 
            position_ids=p_ids
        )

        # Pass through internal encoder
        encoder_outputs = self.encoder.encoder(
            embedding_output,
            attention_mask=extended_attention_mask,
            head_mask=[None] * self.config.num_hidden_layers
        )
        
        sequence_output = encoder_outputs[0]
        logits = self.classifier(self.dropout(sequence_output[:, 0, :]))
        prob = F.softmax(logits, dim=-1)

        if labels is not None:
            loss_fct = CrossEntropyLoss()
            loss = loss_fct(logits, labels)
            return loss, prob
        return prob

In [4]:
class TextDataset(Dataset):
    def __init__(self, tokenizer, args, file_path):
        self.args = args
        self.tokenizer = tokenizer
        self.total_len = args.code_length + args.data_flow_length
        
        with open(file_path, 'r') as f:
            self.lines = f.readlines()
            
    def __len__(self):
        return len(self.lines)

    def _get_char_index(self, code_lines, coord):
        row, col = coord
        char_idx = 0
        for i in range(min(row, len(code_lines))):
            char_idx += len(code_lines[i]) 
        return char_idx + col

    def __getitem__(self, item):
        line = self.lines[item]
        entry = json.loads(line)
        
        code = entry.get('code', '')
        dfg = entry.get('dfg', [])[:self.args.data_flow_length]
        label = int(entry.get('label', 0)) if entry.get('label') is not None else 0

        tokens_obj = self.tokenizer(
            code, 
            max_length=self.args.code_length, 
            truncation=True, 
            padding='max_length',
            return_offsets_mapping=True
        )
        input_ids = tokens_obj['input_ids']
        offsets = tokens_obj['offset_mapping']
        code_lines = code.splitlines(keepends=True)

        # DFG Nodes & Alignment
        dfg_ids = [self.tokenizer.unk_token_id] * len(dfg)
        pos_to_node_idx = {}
        node_to_token_map = {}

        for node_idx, item in enumerate(dfg):
            start_pos, end_pos = item[1][0], item[1][1]
            pos_key = (start_pos[0], start_pos[1], end_pos[0], end_pos[1])
            pos_to_node_idx[pos_key] = node_idx
            
            char_start = self._get_char_index(code_lines, start_pos)
            char_end = self._get_char_index(code_lines, end_pos)
            
            aligned_tokens = []
            for t_idx, (t_start, t_end) in enumerate(offsets):
                if t_start == t_end: continue
                if (t_start >= char_start and t_end <= char_end) or (char_start >= t_start and char_end <= t_end):
                    aligned_tokens.append(t_idx)
            node_to_token_map[node_idx] = aligned_tokens

        # Attention Mask Construction
        attn_mask = np.zeros((self.total_len, self.total_len), dtype=bool)
        c_len = self.args.code_length
        attn_mask[:c_len, :c_len] = True
        
        for node_idx, item in enumerate(dfg):
            abs_node_idx = c_len + node_idx
            for t_idx in node_to_token_map.get(node_idx, []):
                attn_mask[abs_node_idx, t_idx] = True
                attn_mask[t_idx, abs_node_idx] = True
            
            for p_pos in item[4]: # Data flow edges
                p_key = (p_pos[0][0], p_pos[0][1], p_pos[1][0], p_pos[1][1])
                if p_key in pos_to_node_idx:
                    abs_parent_idx = c_len + pos_to_node_idx[p_key]
                    attn_mask[abs_node_idx, abs_parent_idx] = True
                    attn_mask[abs_parent_idx, abs_node_idx] = True
            attn_mask[abs_node_idx, abs_node_idx] = True

        full_input_ids = input_ids + dfg_ids
        p_ids = [i + 2 for i in range(c_len)] + [0] * len(dfg_ids)
        padding_len = self.total_len - len(full_input_ids)
        
        if padding_len > 0:
            full_input_ids += [self.tokenizer.pad_token_id] * padding_len
            p_ids += [1] * padding_len
        
        return {
            'input_ids': torch.tensor(full_input_ids, dtype=torch.long),
            'p_ids': torch.tensor(p_ids, dtype=torch.long),
            'attn_mask': torch.tensor(attn_mask, dtype=torch.float),
            'label': torch.tensor(label, dtype=torch.long)
        }

In [5]:
tokenizer = AutoTokenizer.from_pretrained(args.model_name_or_path, use_fast=True)
full_dataset = TextDataset(tokenizer, args, args.train_file)

def load_entries(filepath):
    with open(filepath, 'r', encoding='utf-8') as f:
        return [json.loads(line) for line in f]

entries = load_entries(args.train_file)
assert len(entries) == len(full_dataset), "Dataset size mismatch between parsed entries and TextDataset"

def infer_source(entry):
    for key in ("source", "dataset", "origin", "project"):
        value = entry.get(key)
        if value is not None and str(value).strip() != "":
            return str(value).strip()
    return "unknown"

def allocate_counts(total_needed, groups, fraction):
    raw = {g: len(v) * fraction for g, v in groups.items()}
    base = {g: int(math.floor(v)) for g, v in raw.items()}
    remainder = total_needed - sum(base.values())
    order = sorted(groups.keys(), key=lambda g: (raw[g] - base[g], len(groups[g])), reverse=True)
    for g in order[:remainder]:
        base[g] += 1
    return base

def stratified_three_way_split(entries, test_ratio=0.10, val_ratio=0.08, seed=42):
    rng = random.Random(seed)
    source_to_indices = defaultdict(list)
    for idx, entry in enumerate(entries):
        source_to_indices[infer_source(entry)].append(idx)

    for indices in source_to_indices.values():
        rng.shuffle(indices)

    total = len(entries)
    target_test = int(round(total * test_ratio))
    target_val = int(round(total * val_ratio))
    target_train = total - target_test - target_val

    test_alloc = allocate_counts(target_test, source_to_indices, test_ratio)
    trainval_groups = {}
    test_indices = []
    for source, indices in source_to_indices.items():
        take = min(test_alloc[source], len(indices))
        test_indices.extend(indices[:take])
        trainval_groups[source] = indices[take:]

    adjusted_val_ratio = val_ratio / (1.0 - test_ratio)
    val_alloc = allocate_counts(target_val, trainval_groups, adjusted_val_ratio)

    val_indices, train_indices = [], []
    for source, indices in trainval_groups.items():
        take = min(val_alloc[source], len(indices))
        val_indices.extend(indices[:take])
        train_indices.extend(indices[take:])

    train_indices = sorted(train_indices)
    val_indices = sorted(val_indices)
    test_indices = sorted(test_indices)

    assert len(train_indices) == target_train, f"Expected train={target_train}, got {len(train_indices)}"
    assert len(val_indices) == target_val, f"Expected val={target_val}, got {len(val_indices)}"
    assert len(test_indices) == target_test, f"Expected test={target_test}, got {len(test_indices)}"
    assert set(train_indices).isdisjoint(val_indices)
    assert set(train_indices).isdisjoint(test_indices)
    assert set(val_indices).isdisjoint(test_indices)

    return train_indices, val_indices, test_indices

train_indices, val_indices, test_indices = stratified_three_way_split(
    entries,
    test_ratio=args.test_ratio,
    val_ratio=args.val_ratio,
    seed=args.seed,
)

train_dataset = Subset(full_dataset, train_indices)
val_dataset = Subset(full_dataset, val_indices)
test_dataset = Subset(full_dataset, test_indices)

os.makedirs(args.output_dir, exist_ok=True)
with open(args.split_indices_path, 'w', encoding='utf-8') as f:
    json.dump({
        'seed': args.seed,
        'train_indices': train_indices,
        'val_indices': val_indices,
        'test_indices': test_indices,
    }, f)

def source_counts(indices):
    counts = Counter(infer_source(entries[i]) for i in indices)
    return dict(sorted(counts.items()))

split_summary = {
    'total': len(entries),
    'train': len(train_indices),
    'val': len(val_indices),
    'test': len(test_indices),
    'train_source_counts': source_counts(train_indices),
    'val_source_counts': source_counts(val_indices),
    'test_source_counts': source_counts(test_indices),
}
with open(args.split_summary_path, 'w', encoding='utf-8') as f:
    json.dump(split_summary, f, indent=2)

print('Dataset Split Results')
print(f"Total: {split_summary['total']}")
print(f"Train: {split_summary['train']}")
print(f"Val: {split_summary['val']}")
print(f"Test: {split_summary['test']}")
print('Train source counts:', split_summary['train_source_counts'])
print('Val source counts:', split_summary['val_source_counts'])
print('Test source counts:', split_summary['test_source_counts'])


INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/graphcodebert-base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/graphcodebert-base/2b0488a7bb0eefc7041f1bb2cad1ab26b0da269d/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/resolve-cache/models/microsoft/graphcodebert-base/2b0488a7bb0eefc7041f1bb2cad1ab26b0da269d/config.json "HTTP/1.1 200 OK"


config.json:   0%|          | 0.00/539 [00:00<?, ?B/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/graphcodebert-base/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/graphcodebert-base/2b0488a7bb0eefc7041f1bb2cad1ab26b0da269d/tokenizer_config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/resolve-cache/models/microsoft/graphcodebert-base/2b0488a7bb0eefc7041f1bb2cad1ab26b0da269d/tokenizer_config.json "HTTP/1.1 200 OK"


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/microsoft/graphcodebert-base/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/microsoft/graphcodebert-base/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/graphcodebert-base/resolve/main/vocab.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/graphcodebert-base/2b0488a7bb0eefc7041f1bb2cad1ab26b0da269d/vocab.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/resolve-cache/models/microsoft/graphcodebert-base/2b0488a7bb0eefc7041f1bb2cad1ab26b0da269d/vocab.json "HTTP/1.1 200 OK"


...

merges.txt: 0.00B [00:00, ?B/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/graphcodebert-base/resolve/main/tokenizer.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/graphcodebert-base/resolve/main/added_tokens.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/graphcodebert-base/resolve/main/special_tokens_map.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/graphcodebert-base/2b0488a7bb0eefc7041f1bb2cad1ab26b0da269d/special_tokens_map.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/resolve-cache/models/microsoft/graphcodebert-base/2b0488a7bb0eefc7041f1bb2cad1ab26b0da269d/special_tokens_map.json "HTTP/1.1 200 OK"


special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/graphcodebert-base/resolve/main/chat_template.jinja "HTTP/1.1 404 Not Found"


Dataset Split Results
Total: 199960
Train: 163967
Val: 15997
Test: 19996
Train source counts: {'unknown': 163967}
Val source counts: {'unknown': 15997}
Test source counts: {'unknown': 19996}


In [6]:
def evaluate(model, dataset, args, tag="Eval", quiet=False):
    dataloader = DataLoader(
        dataset,
        sampler=SequentialSampler(dataset),
        batch_size=args.eval_batch_size,
        num_workers=args.num_workers,
    )
    model.eval()
    all_probs, all_labels = [], []

    for batch in tqdm(dataloader, desc=tag, disable=quiet):
        input_ids = batch['input_ids'].to(args.device)
        p_ids = batch['p_ids'].to(args.device)
        attn_mask = batch['attn_mask'].to(args.device)
        labels = batch['label'].to(args.device)

        with torch.no_grad():
            with autocast(device_type='cuda', enabled=torch.cuda.is_available()):
                probs = model(input_ids=input_ids, p_ids=p_ids, attn_mask=attn_mask)

        all_probs.append(probs.detach().cpu().numpy())
        all_labels.append(labels.detach().cpu().numpy())

    all_probs = np.concatenate(all_probs, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)
    preds = (all_probs[:, 1] >= 0.5).astype(int)

    acc = accuracy_score(all_labels, preds)
    rocauc = roc_auc_score(all_labels, all_probs[:, 1])
    prauc = average_precision_score(all_labels, all_probs[:, 1])
    tn, fp, fn, tp = confusion_matrix(all_labels, preds).ravel()

    if not quiet:
        logger.info(f"--- {tag} Results ---")
        logger.info(f"Accuracy {acc:.4f}")
        logger.info(f"ROC-AUC {rocauc:.4f} | PR-AUC {prauc:.4f}")
        logger.info(f"FN {fn} | FP {fp}")

    return {
        'probs': all_probs,
        'labels': all_labels,
        'acc': acc,
        'rocauc': rocauc,
        'prauc': prauc,
        'fn': int(fn),
        'fp': int(fp),
    }


def train_model(model, train_dataset, val_dataset, args):
    train_dataloader = DataLoader(
        train_dataset,
        sampler=RandomSampler(train_dataset),
        batch_size=args.train_batch_size,
        drop_last=True,
        num_workers=args.num_workers,
    )

    optimizer = AdamW(model.parameters(), lr=args.learning_rate, eps=1e-8)
    total_steps = len(train_dataloader) * args.num_train_epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(total_steps * 0.1),
        num_training_steps=total_steps,
    )
    scaler = GradScaler('cuda', enabled=torch.cuda.is_available())

    best_val_acc = -1.0
    best_epoch = 0
    patience_counter = 0
    history = []

    os.makedirs(args.output_dir, exist_ok=True)

    for epoch in range(args.num_train_epochs):
        model.train()
        total_loss = 0.0
        logger.info(f"Starting Epoch {epoch + 1}/{args.num_train_epochs}")
        progress_bar = tqdm(train_dataloader, desc=f"Epoch {epoch + 1}")

        for batch in progress_bar:
            input_ids = batch['input_ids'].to(args.device)
            p_ids = batch['p_ids'].to(args.device)
            attn_mask = batch['attn_mask'].to(args.device)
            labels = batch['label'].to(args.device)

            optimizer.zero_grad(set_to_none=True)
            with autocast(device_type='cuda', enabled=torch.cuda.is_available()):
                loss, _ = model(input_ids=input_ids, p_ids=p_ids, attn_mask=attn_mask, labels=labels)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), args.max_grad_norm)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            total_loss += loss.item()
            progress_bar.set_postfix(loss=f"{loss.item():.4f}")

        avg_train_loss = total_loss / len(train_dataloader)
        logger.info(f"Epoch {epoch + 1} Average Train Loss {avg_train_loss:.4f}")

        val_metrics = evaluate(model, val_dataset, args, tag=f"Validation Epoch {epoch + 1}")
        val_acc = val_metrics['acc']
        history.append({
            'epoch': epoch + 1,
            'train_loss': avg_train_loss,
            'val_acc': val_acc,
            'val_rocauc': val_metrics['rocauc'],
            'val_prauc': val_metrics['prauc'],
            'val_fn': val_metrics['fn'],
            'val_fp': val_metrics['fp'],
        })

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_epoch = epoch + 1
            patience_counter = 0
            torch.save(model.state_dict(), args.best_model_path)
            logger.info(f"New Best Model Saved at Epoch {best_epoch} with Val Acc {best_val_acc:.4f}")
        else:
            patience_counter += 1
            logger.info(f"No improvement. Patience {patience_counter}/{args.patience}")
            if patience_counter >= args.patience:
                logger.info("Early stopping triggered. Halting training.")
                break

    return best_epoch, best_val_acc, history


In [7]:
config = RobertaConfig.from_pretrained(args.model_name_or_path)
config.num_labels = 2
encoder = RobertaModel.from_pretrained(args.model_name_or_path, config=config)
model = Model(encoder, config)
model.to(args.device)

best_epoch, best_val_acc, train_history = train_model(model, train_dataset, val_dataset, args)

model.load_state_dict(torch.load(args.best_model_path, map_location=args.device))

test_metrics = evaluate(model, test_dataset, args, tag="Final Test", quiet=False)

probs = test_metrics['probs']
labels = test_metrics['labels']
acc = test_metrics['acc']
rocauc = test_metrics['rocauc']
prauc = test_metrics['prauc']
fn = test_metrics['fn']
fp = test_metrics['fp']

np.save('/kaggle/working/test_probs.npy', probs)
np.save('/kaggle/working/test_labels.npy', labels)

history_path = '/kaggle/working/graphcodebert_training_history.json'
with open(history_path, 'w', encoding='utf-8') as f:
    json.dump(train_history, f, indent=2)

out_path = '/kaggle/working/graphcodebert_results.txt'
with open(out_path, 'w', encoding='utf-8') as f:
    f.write('Model: GraphCodeBERT + DFG\n')
    f.write('Split: stratified train/val/test\n')
    f.write(f'Seed: {args.seed}\n')
    f.write(f'Train size: {len(train_dataset)}\n')
    f.write(f'Val size: {len(val_dataset)}\n')
    f.write(f'Test size: {len(test_dataset)}\n')
    f.write(f'Epoch ceiling: {args.num_train_epochs}\n')
    f.write(f'Patience: {args.patience}\n')
    f.write(f'Best epoch: {best_epoch}\n')
    f.write(f'Best val accuracy: {best_val_acc:.4f}\n')
    f.write(f'Test Accuracy: {acc:.4f}\n')
    f.write(f'Test ROC-AUC: {rocauc:.4f}\n')
    f.write(f'Test PR-AUC: {prauc:.4f}\n')
    f.write(f'FN: {fn}\n')
    f.write(f'FP: {fp}\n')
    f.write(f'Split indices file: {args.split_indices_path}\n')
    f.write(f'Split summary file: {args.split_summary_path}\n')
    f.write(f'Training history file: {history_path}\n')

print(f'Saved results to {out_path}')
print(f'Best epoch selected by early stopping: {best_epoch}')


INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/graphcodebert-base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/graphcodebert-base/2b0488a7bb0eefc7041f1bb2cad1ab26b0da269d/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/graphcodebert-base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/graphcodebert-base/2b0488a7bb0eefc7041f1bb2cad1ab26b0da269d/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/graphcodebert-base/resolve/main/adapter_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/graphcodebert-base/resolve/main/model.safetensors "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/graphcodebert-base/re

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/graphcodebert-base/resolve/main/model.safetensors "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/microsoft/graphcodebert-base "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/microsoft/graphcodebert-base/commits/main "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/microsoft/graphcodebert-base/discussions?p=0 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/microsoft/graphcodebert-base/commits/refs%2Fpr%2F8 "HTTP/1.1 200 OK"


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/graphcodebert-base/resolve/refs%2Fpr%2F8/model.safetensors.index.json "HTTP/1.1 404 Not Found"
RobertaModel LOAD REPORT from: microsoft/graphcodebert-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.decoder.bias            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. C

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

INFO:__main__:Starting Epoch 1/5

Epoch 1:   0%|          | 0/10247 [00:00<?, ?it/s]/tmp/ipykernel_21/4192354373.py:96: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()

Epoch 1: 100%|██████████| 10247/10247 [1:16:44<00:00,  2.23it/s, loss=0.4163]
INFO:__main__:Epoch 1 Average Train Loss 0.3481
Validation Epoch 1: 100%|██████████| 500/500 [02:11<00:00,  3.80it/s]
INFO:__main__:--- Validation Epoch 1 Results ---
INFO:__main__:Accuracy 0.8590
INFO:__main__:ROC-AUC 0.9470 | PR-AUC 0.9487
INFO:__main__:FN 1591 | FP 665
INFO:__main__:New Best Model Saved at Epoch 1 with Val Acc 0.8590
INFO:__main__:Starting Epoch 2/5
Epoch 2: 100%|██████

Saved results to /kaggle/working/graphcodebert_results.txt
Best epoch selected by early stopping: 5
